In [1]:
# 1. 导入库
import numpy as np
import pandas as pd

# 2. 导入数据
df = pd.read_csv('../data/sleep.csv')
df.describe().round(2)

,person_id,age,sleep_duration,sleep_quality,physical_activity_level,stress_level,heart_rate,daily_steps
count,400.00,400.00,400.00,400.00,400.00,400.00,400.00,400.00
mean,200.50,39.95,8.04,6.13,64.98,5.47,75.99,11076.51
std,115.61,14.04,2.39,1.98,32.30,2.81,15.10,5364.79
min,1.00,18.00,4.10,1.00,10.00,1.00,50.00,2067.00
25%,100.75,29.00,5.90,4.70,35.00,3.00,63.00,6165.25
50%,200.50,40.00,8.20,6.10,65.50,5.00,77.00,11785.50
75%,300.25,49.00,10.12,7.43,94.00,8.00,90.00,15878.00
max,400.00,90.00,12.00,10.00,120.00,10.00,100.00,19958.00


In [2]:
df.head()

,person_id,gender,age,occupation,sleep_duration,sleep_quality,physical_activity_level,stress_level,bmi_category,blood_pressure,heart_rate,daily_steps,sleep_disorder
0,1,Male,29,Manual Labor,7.4,7.0,41,7,Obese,124/70,91,8539,NaN
1,2,Female,43,Retired,4.2,4.9,41,5,Obese,131/86,81,18754,NaN
2,3,Male,44,Retired,6.1,6.0,107,4,Underweight,122/70,81,2857,NaN
3,4,Male,29,Office Worker,8.3,10.0,20,10,Obese,124/72,55,6886,NaN
4,5,Male,67,Retired,9.1,9.5,19,4,Overweight,133/78,97,14945,Insomnia


In [3]:
# 3. 数据清洗
# 查看有无缺损
df.isna().sum()
# 查看缺损列其他数据情况
df['sleep_disorder'].value_counts()
# 按列丢弃
df = df.drop(columns='sleep_disorder')
df.isna().sum()

person_id                  0
gender                     0
age                        0
occupation                 0
sleep_duration             0
sleep_quality              0
physical_activity_level    0
stress_level               0
bmi_category               0
blood_pressure             0
heart_rate                 0
daily_steps                0
dtype: int64

In [4]:
df.head()

,person_id,gender,age,occupation,sleep_duration,sleep_quality,physical_activity_level,stress_level,bmi_category,blood_pressure,heart_rate,daily_steps
0,1,Male,29,Manual Labor,7.4,7.0,41,7,Obese,124/70,91,8539
1,2,Female,43,Retired,4.2,4.9,41,5,Obese,131/86,81,18754
2,3,Male,44,Retired,6.1,6.0,107,4,Underweight,122/70,81,2857
3,4,Male,29,Office Worker,8.3,10.0,20,10,Obese,124/72,55,6886
4,5,Male,67,Retired,9.1,9.5,19,4,Overweight,133/78,97,14945


In [5]:
# 4. 数据特征构造
# 能转类别的转类别
df.head()
print(df['gender'].dtype, df['occupation'].dtype, df['bmi_category'].dtype)
df['gender'] = df['gender'].astype('category')
df['occupation'] = df['occupation'].astype('category')
df['bmi_category'] = df['bmi_category'].astype('category')
print(df['gender'].dtype, df['occupation'].dtype, df['bmi_category'].dtype)

str str str
category category category


In [6]:
# 血压分高血压，低血压
df[['high','low']] = df['blood_pressure'].str.split('/',expand=True)

In [7]:
# 睡眠质量分箱 quality_level - perfect good bad
df['quality_level'] = pd.cut(df['sleep_quality'], bins=3, labels=['perfect','good', 'bad'])
df.head()

,person_id,gender,age,occupation,sleep_duration,sleep_quality,physical_activity_level,stress_level,bmi_category,blood_pressure,heart_rate,daily_steps,high,low,quality_level
0,1,Male,29,Manual Labor,7.4,7.0,41,7,Obese,124/70,91,8539,124,70,good
1,2,Female,43,Retired,4.2,4.9,41,5,Obese,131/86,81,18754,131,86,good
2,3,Male,44,Retired,6.1,6.0,107,4,Underweight,122/70,81,2857,122,70,good
3,4,Male,29,Office Worker,8.3,10.0,20,10,Obese,124/72,55,6886,124,72,bad
4,5,Male,67,Retired,9.1,9.5,19,4,Overweight,133/78,97,14945,133,78,bad


In [8]:
# 年龄分箱 age_level - youth middle older
df['age_level'] = pd.cut(df['age'], bins=[0,18,60,np.inf], labels=['youth','adult', 'older'])
df.head()

,person_id,gender,age,occupation,sleep_duration,sleep_quality,physical_activity_level,stress_level,bmi_category,blood_pressure,heart_rate,daily_steps,high,low,quality_level,age_level
0,1,Male,29,Manual Labor,7.4,7.0,41,7,Obese,124/70,91,8539,124,70,good,adult
1,2,Female,43,Retired,4.2,4.9,41,5,Obese,131/86,81,18754,131,86,good,adult
2,3,Male,44,Retired,6.1,6.0,107,4,Underweight,122/70,81,2857,122,70,good,adult
3,4,Male,29,Office Worker,8.3,10.0,20,10,Obese,124/72,55,6886,124,72,bad,adult
4,5,Male,67,Retired,9.1,9.5,19,4,Overweight,133/78,97,14945,133,78,bad,older


In [9]:
# 5. 数据统计
# 查看 bmi 和 睡眠质量 的关系
df.groupby(['age_level']).agg({
    'sleep_quality': ['mean', 'median', 'max', 'min'],
    'sleep_duration': ['mean', 'median', 'max', 'min']
}).round(2)

sleep_quality                   sleep_duration                  
                   mean median   max  min           mean median   max  min
age_level                                                                 
youth              6.26    6.3   8.5  3.4           7.48   8.05  10.9  4.1
adult              6.13    6.1  10.0  1.0           8.09   8.20  12.0  4.1
older              5.93    5.6  10.0  1.0           8.00   8.70  12.0  4.2

In [10]:
df.groupby(['bmi_category']).agg({
    'sleep_quality': ['mean', 'median', 'max', 'min'],
    'sleep_duration': ['mean', 'median', 'max', 'min']
}).round(2)

sleep_quality                   sleep_duration                  
                      mean median   max  min           mean median   max  min
bmi_category                                                                 
Normal                6.34    6.5  10.0  1.0           7.79    7.6  12.0  4.1
Obese                 6.19    6.2  10.0  2.0           8.07    8.2  12.0  4.1
Overweight            6.10    6.1  10.0  1.0           8.27    8.6  12.0  4.1
Underweight           5.90    6.0  10.0  1.7           7.98    8.1  11.9  4.2

In [11]:
df.groupby(['age_level','bmi_category']).agg({
    'sleep_quality': ['mean', 'median', 'max', 'min'],
    'sleep_duration': ['mean', 'median', 'max', 'min']
}).round(2)

sleep_quality                   sleep_duration         \
                                mean median   max  min           mean median   
age_level bmi_category                                                         
youth     Normal                6.75   6.75   7.9  5.4           7.43   7.40   
          Obese                 7.34   8.20   8.5  5.4           7.34   6.10   
          Overweight            5.23   5.30   6.6  3.4           7.13   7.75   
          Underweight           6.07   6.10   7.5  3.7           7.75   8.50   
adult     Normal                6.44   6.70  10.0  1.0           7.92   7.80   
          Obese                 6.17   6.15  10.0  2.0           8.14   8.30   
          Overweight            6.11   6.20  10.0  1.0           8.32   8.50   
          Underweight           5.87   5.95  10.0  1.7           7.95   7.80   
older     Normal                5.56   5.15  10.0  3.3           7.19   6.60   
          Obese                 5.44   4.70   8.2  3.6           7.56   6.20   
          Overweight            6.41   6.15  10.0  1.0           8.49   8.95   
          Underweight           5.97   6.20  10.0  1.7           9.70   9.60   

                                   
                         max  min  
age_level bmi_category             
youth     Normal        10.1  4.2  
          Obese         10.8  5.0  
          Overweight     9.5  4.1  
          Underweight   10.9  4.7  
adult     Normal        12.0  4.1  
          Obese         12.0  4.1  
          Overweight    11.9  4.1  
          Underweight   11.9  4.2  
older     Normal        10.5  4.3  
          Obese         11.6  4.2  
          Overweight    12.0  5.8  
          Underweight   10.5  9.0